In [1]:
import os
from openai import OpenAI
import requests
import json
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown,display,update_display

In [2]:
load_dotenv(override=True)
OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
if OPEN_API_KEY and OPEN_API_KEY.startswith("sk-proj-") and len(OPEN_API_KEY)>10:
    print("API key exists in env")
else:
    print("There might be a problem with your API key")
model = "gpt-5-nano"
openai = OpenAI()

API key exists in env


In [3]:
def website_scrapper(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "no title found"
    root = soup.body if soup.body is not None else soup
    for tag in root.find_all(["script", "style", "img", "input", "link"]):
        tag.decompose()
    text = root.get_text(separator=" ", strip=True)[:2_000]
    return title, text

In [4]:
def get_links(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [5]:
# links = get_links("https://huggingface.co")
# links

In [6]:
#title,text = website_scrapper("https://huggingface.co")

In [7]:

# from httpx import get


# openai = OpenAI()
# title,text = website_scrapper("https://www.cnn.com")
# messages = [{"role":"system","content":"you are a summarizer"},{"role":"user","content":"Can you summarize"+text}]
# response = openai.chat.completions.create(model="gpt-5-nano",messages=messages)
# response.choices[0].message.content

In [8]:
model_system_prompt="""
You are provided with a list of links found in a webpage.
you are able to decide which links would be most relevant to include in a brochure about a company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
you should respond in json as in this example:"""
{
    "useful_links": [
        {"company_website": "https://www.cnn.com", "type": "company website"},
        {
            "company_linkedin": "https://www.linkedin.com/company/cnn",
            "type": "company linkedin",
        },
        {"company_twitter": "https://twitter.com/cnn", "type": "company twitter"},
        {
            "company_facebook": "https://www.facebook.com/cnn",
            "type": "company facebook",
        },
        {
            "company_instagram": "https://www.instagram.com/cnn",
            "type": "company instagram",
        },
        {"company_youtube": "https://www.youtube.com/cnn", "type": "company youtube"},
    ]
}

{'useful_links': [{'company_website': 'https://www.cnn.com',
   'type': 'company website'},
  {'company_linkedin': 'https://www.linkedin.com/company/cnn',
   'type': 'company linkedin'},
  {'company_twitter': 'https://twitter.com/cnn', 'type': 'company twitter'},
  {'company_facebook': 'https://www.facebook.com/cnn',
   'type': 'company facebook'},
  {'company_instagram': 'https://www.instagram.com/cnn',
   'type': 'company instagram'},
  {'company_youtube': 'https://www.youtube.com/cnn',
   'type': 'company youtube'}]}

In [9]:
def get_user_prompt(url:str):
    model_user_prompt=f"""Here is list of links on the website {url}
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of service , Privacy , email links.
    Links (Some might be relative links) """
    links = get_links(url)
    model_user_prompt += "\n".join(links)
    return model_user_prompt

In [10]:
def get_relevant_links(url: str):
    print(f"Select relevant links for {url} using model {model}")
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": model_system_prompt},
            {"role": "user", "content": get_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    if not result:
        raise ValueError("Model returned empty content; check model name and API response.")
    links = json.loads(result)
    return links


In [11]:
def _link_entries_from_model(parsed: dict):
    """Normalize model JSON: links may be under 'links' or 'useful_links', items may be URLs or dicts."""
    raw = parsed.get("links") or parsed.get("useful_links") or []
    out = []
    for item in raw:
        if isinstance(item, str):
            out.append((item, "link"))
        elif isinstance(item, dict):
            if "url" in item and isinstance(item["url"], str):
                out.append((item["url"], item.get("type", "link")))
            else:
                for k, v in item.items():
                    if k != "type" and isinstance(v, str) and v.startswith("http"):
                        out.append((v, item.get("type", k)))
                        break
    return out


def fetch_page_relevant_links(url):
    _title, landing_text = website_scrapper(url)
    relevant_links = get_relevant_links(url)
    result = f"## Landing page:\n\n{landing_text}\n ## relevant Links:\n"
    for link_url, link_type in _link_entries_from_model(relevant_links):
        result += f"\n\n ## Link : {link_type}\n"
        _t, page_text = website_scrapper(link_url)
        result += page_text
    return result


In [12]:
brochure_system_prompt = """You are an assistant that analyzes the contents of several relevant pages from a company website and creates a short brochure about a company for prospective customers, investors and recruits. Respond in markdown without code blocks. Include details of company culture, customers and carrers/jobs if you have a information. """

In [13]:
def get_brochure_user_prompt(companyname, url):
    user_prompt = f""" you are looking at a comapany called: {companyname} 
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n"""
    user_prompt += fetch_page_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [14]:
#get_brochure_user_prompt("Hugging face", "https://huggingface.co")

In [15]:
def create_brochure(companyname,url):
    response = openai.chat.completions.create(model = "gpt-4.1-mini",
    messages=[
        {"role":"system","content":brochure_system_prompt},
        {"role":"user","content":get_brochure_user_prompt(companyname,url)}
    ])
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace","https://huggingface.co")

In [ ]:
#create_brochure("Vittal manikonda","https://www.linkedin.com/in/vittal-manikonda")

Select relevant links for https://www.linkedin.com/in/vittal-manikonda using model gpt-5-nano


# Vittal Manikonda

Welcome to Vittal Manikonda, a company dedicated to innovation, excellence, and creating impactful solutions for our clients. We pride ourselves on delivering high-quality products and services tailored to meet the specific needs of our diverse customer base.

## About Us
At Vittal Manikonda, we believe in combining cutting-edge technology with a customer-centric approach to drive success. Our team of passionate professionals is committed to continuous improvement and pushing the boundaries to provide unmatched value.

## Our Customers
We serve a broad spectrum of industries, offering customized solutions that help businesses thrive in a competitive landscape. Our customers include startups looking for agile development, SMEs aiming for digital transformation, and large enterprises that require robust, scalable systems.

## Company Culture
Our culture is built around collaboration, innovation, and integrity. We foster an environment where creativity and ideas can flourish, encouraging our employees to take initiative and embrace challenges. Diversity and inclusion are core to our workplace, ensuring that every voice is heard and respected.

## Careers
Join Vittal Manikonda and be part of a growing team that values professional growth and personal development. We offer exciting career opportunities in various domains including software development, project management, marketing, and customer support. Our employees enjoy flexible work arrangements, continuous learning opportunities, and a supportive work environment.

### Why Work With Us?
- Dynamic and inclusive work culture  
- Opportunities for career advancement  
- Training and skill development programs  
- Work-life balance initiatives  

---

For more information or to explore career opportunities, visit our website or contact our HR team directly.

**Vittal Manikonda — Driving Innovation, Delivering Excellence**